## Blocks World

This notebook models the classic **blocks world** domain with three blocks in the unified-planning library.

The world has a table, a set of blocks and a robotic arm (gripper). The arm can hold at most one block at a time and can only pick up a block that has nothing on top of it (a *clear* block). We use the standard four operators:

 - `pick_up(x)` – pick up block `x` from the table,
 - `put_down(x)` – put the held block `x` on the table,
 - `stack(x, y)` – put the held block `x` on top of block `y`,
 - `unstack(x, y)` – pick up block `x` from the top of block `y`.

As the problem instance we use the **Sussman anomaly**:

```
 Initial state        Goal state
                         A
   C                     B
   A   B                 C
 ---------           ---------
```

### Setup

In [19]:
#%pip install "unified-planning[pyperplan,fast-downward]"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: C:\Users\limek\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [20]:
from unified_planning.shortcuts import *

# Do not print the credits of the planning engines every time we call them
get_environment().credits_stream = None

### Domain

#### Types and fluents

There is a single user type, `Block`. The state of the world is described by the following boolean fluents:

 - `on(x, y)` – block `x` lies directly on block `y`,
 - `on_table(x)` – block `x` lies on the table,
 - `clear(x)` – there is nothing on top of block `x`,
 - `holding(x)` – the arm holds block `x`,
 - `hand_empty` – the arm holds nothing.

In [21]:
Block = UserType("Block")

on = Fluent("on", BoolType(), x=Block, y=Block)
on_table = Fluent("on_table", BoolType(), x=Block)
clear = Fluent("clear", BoolType(), x=Block)
holding = Fluent("holding", BoolType(), x=Block)
hand_empty = Fluent("hand_empty", BoolType())

#### Actions

Each action lists its preconditions and its effects. All fluents not mentioned in the effects keep their values (the frame assumption).

In [22]:
pick_up = InstantaneousAction("pick_up", x=Block)
x = pick_up.parameter("x")
pick_up.add_precondition(clear(x))
pick_up.add_precondition(on_table(x))
pick_up.add_precondition(hand_empty())
pick_up.add_effect(on_table(x), False)
pick_up.add_effect(clear(x), False)
pick_up.add_effect(hand_empty(), False)
pick_up.add_effect(holding(x), True)

put_down = InstantaneousAction("put_down", x=Block)
x = put_down.parameter("x")
put_down.add_precondition(holding(x))
put_down.add_effect(hand_empty(), True)
put_down.add_effect(holding(x), False)
put_down.add_effect(on_table(x), True)
put_down.add_effect(clear(x), True)

stack = InstantaneousAction("stack", x=Block, y=Block)
x = stack.parameter("x")
y = stack.parameter("y")
stack.add_precondition(holding(x))
stack.add_precondition(clear(y))
stack.add_effect(holding(x), False)
stack.add_effect(hand_empty(), True)
stack.add_effect(on(x,y), True)
stack.add_effect(clear(y), False)
stack.add_effect(clear(x), True)


unstack = InstantaneousAction("unstack", x=Block, y=Block)
x = unstack.parameter("x")
y = unstack.parameter("y")
unstack.add_precondition(clear(x))
unstack.add_precondition(on(x,y))
unstack.add_precondition(hand_empty())
unstack.add_effect(on(x,y), False)
unstack.add_effect(hand_empty(), False)
unstack.add_effect(holding(x), True)
unstack.add_effect(clear(y), True)
unstack.add_effect(clear(x), False)

print(stack)

action stack(Block x, Block y) {
    preconditions = [
      holding(x)
      clear(y)
    ]
    effects = [
      holding(x) := false
      hand_empty := true
      on(x, y) := true
      clear(y) := false
      clear(x) := true
    ]
  }


### Problem: the Sussman anomaly

We create three blocks and describe the initial state and the goal. Fluents not set explicitly default to `False`.

In [23]:
problem = Problem("blocks_world")
for f in (on, on_table, clear, holding, hand_empty):
    problem.add_fluent(f, default_initial_value=False)
problem.add_actions([pick_up, put_down, stack, unstack])

A, B, C = (Object(name, Block) for name in "ABC")
problem.add_objects([A, B, C])

# Initial state: C is on A, A and B are on the table, the arm is empty
problem.set_initial_value(on(C, A), True)
problem.set_initial_value(on_table(A), True)
problem.set_initial_value(clear(C), True)
problem.set_initial_value(clear(B), True)
problem.set_initial_value(on_table(B), True)
problem.set_initial_value(hand_empty(), True)


# Goal: the tower A on B on C
problem.add_goal(on(A, B))
problem.add_goal(on(B, C))

print(problem)

problem name = blocks_world

types = [Block]

fluents = [
  bool on[x=Block, y=Block]
  bool on_table[x=Block]
  bool clear[x=Block]
  bool holding[x=Block]
  bool hand_empty
]

actions = [
  action pick_up(Block x) {
    preconditions = [
      clear(x)
      on_table(x)
      hand_empty
    ]
    effects = [
      on_table(x) := false
      clear(x) := false
      hand_empty := false
      holding(x) := true
    ]
  }
  action put_down(Block x) {
    preconditions = [
      holding(x)
    ]
    effects = [
      hand_empty := true
      holding(x) := false
      on_table(x) := true
      clear(x) := true
    ]
  }
  action stack(Block x, Block y) {
    preconditions = [
      holding(x)
      clear(y)
    ]
    effects = [
      holding(x) := false
      hand_empty := true
      on(x, y) := true
      clear(y) := false
      clear(x) := true
    ]
  }
  action unstack(Block x, Block y) {
    preconditions = [
      clear(x)
      on(x, y)
      hand_empty
    ]
    effects = [
      

### Solving the problem

We let the library pick a suitable planner for this problem kind with `OneshotPlanner(problem_kind=...)`.

In [24]:
with OneshotPlanner(name="pyperplan") as planner:
    result = planner.solve(problem)
    print(f"Planner: {planner.name}, status: {result.status.name}")
    plan = result.plan
    print(plan)

Planner: Pyperplan, status: SOLVED_SATISFICING
SequentialPlan:
    unstack(C, A)
    put_down(C)
    pick_up(B)
    stack(B, C)
    pick_up(A)
    stack(A, B)


To get a plan with the minimum number of actions, we add a quality metric (minimize the plan length) and require an optimality guarantee from the planner. The anomaly needs 6 actions: C has to be moved to the table first, before B can be put on C and A on B.

In [25]:
problem.add_quality_metric(MinimizeSequentialPlanLength())

with OneshotPlanner(
    name="pyperplan",
    optimality_guarantee=OptimalityGuarantee.SOLVED_OPTIMALLY,
) as planner:
    result = planner.solve(problem)
    print(f"Planner: {planner.name}, status: {result.status.name}")
    plan = result.plan
    print(plan)

Planner: Pyperplan, status: SOLVED_SATISFICING
SequentialPlan:
    unstack(C, A)
    put_down(C)
    pick_up(B)
    stack(B, C)
    pick_up(A)
    stack(A, B)


### Simulating the plan

We use the `SequentialSimulator` to replay the plan step by step and draw the towers in each intermediate state.

In [26]:
blocks = [A, B, C]

def draw(state):
    """Return an ASCII drawing of the towers in a given state."""
    def value(fluent):
        return state.get_value(fluent).bool_constant_value()

    towers = []
    for base in blocks:
        if value(on_table(base)):
            tower = [base]
            while True:
                above = [b for b in blocks if value(on(b, tower[-1]))]
                if not above:
                    break
                tower.append(above[0])
            towers.append(tower)
    held = [b.name for b in blocks if value(holding(b))]
    height = max(len(t) for t in towers) if towers else 0
    lines = [f"  hand: {held[0] if held else '-'}"]
    for level in reversed(range(height)):
        lines.append("  " + " ".join(t[level].name if level < len(t) else " " for t in towers))
    lines.append("  " + "-" * (2 * len(towers) - 1 if towers else 1))
    return "\n".join(lines)

with SequentialSimulator(problem) as simulator:
    state = simulator.get_initial_state()
    print("initial state")
    print(draw(state))
    for step, action in enumerate(plan.actions, start=1):
        state = simulator.apply(state, action)
        print(f"\n{step}. {action}")
        print(draw(state))
    print("\ngoal reached:", simulator.is_goal(state))

initial state
  hand: -
  C  
  A B
  ---

1. unstack(C, A)
  hand: C
  A B
  ---

2. put_down(C)
  hand: -
  A B C
  -----

3. pick_up(B)
  hand: B
  A C
  ---

4. stack(B, C)
  hand: -
    B
  A C
  ---

5. pick_up(A)
  hand: A
  B
  C
  -

6. stack(A, B)
  hand: -
  A
  B
  C
  -

goal reached: True


### Exporting to PDDL

The same problem can be exported to PDDL, e.g. to be solved by any external PDDL planner.

In [27]:
from unified_planning.io import PDDLWriter

writer = PDDLWriter(problem)
print(writer.get_domain())
print(writer.get_problem())

(define (domain blocks_world-domain)
 (:requirements :strips :typing :action-costs)
 (:types block)
 (:predicates 
             (on ?x - block ?y - block)
             (on_table ?x - block)
             (clear ?x - block)
             (holding ?x - block)
             (hand_empty)
 )
 (:functions 
             (total-cost)
 )
 (:action pick_up
  :parameters ( ?x - block)
  :precondition (and (clear ?x) (on_table ?x) (hand_empty))
  :effect (and (not (on_table ?x)) (not (clear ?x)) (not (hand_empty)) (holding ?x) (increase (total-cost) 1)))
 (:action put_down
  :parameters ( ?x - block)
  :precondition (and (holding ?x))
  :effect (and (hand_empty) (not (holding ?x)) (on_table ?x) (clear ?x) (increase (total-cost) 1)))
 (:action stack
  :parameters ( ?x - block ?y - block)
  :precondition (and (holding ?x) (clear ?y))
  :effect (and (not (holding ?x)) (hand_empty) (on ?x ?y) (not (clear ?y)) (clear ?x) (increase (total-cost) 1)))
 (:action unstack
  :parameters ( ?x - block ?y - block)


In [29]:
import logging
import sys

# Wymuszamy wypisywanie wszystkich logów diagnostycznych
logging.basicConfig(stream=sys.stdout, level=logging.DEBUG, force=True)

print("--- START DEBUGOWANIA ---")
with OneshotPlanner(name="fast-downward") as planner:
    result = planner.solve(problem)
    print(f"Status: {result.status.name}")

--- START DEBUGOWANIA ---
Status: INTERNAL_ERROR
